# 07 — Factor annotation: supertypes and gene programs

For each live DRVI direction:
1. **Supertype specificity** — which SST supertypes activate it most (from pseudobulk means).
2. **Top genes** — out-of-distribution traversal scores from the decoder.
3. **GSEA enrichment** — ORA against GO BP / MF / CC and MSigDB Hallmark.
4. **Summary table** — one row per direction linking all three.


In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import anndata as ad
import seaborn as sns
import gseapy as gp

import sst_drvi as S

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)

logger = logging.getLogger("annotation")
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
    logger.setLevel(logging.INFO)


## §1 — Setup

Prefers converged covariate refit, falls back gracefully.

In [ ]:
CANDIDATES = ["sst_k64_covar_long", "sst_k64_covar", "sst_k64"]
MODEL = "drvi"

RUN = None
for candidate in CANDIDATES:
    embed_path = S.EMBED_DIR / f"{candidate}_{MODEL}_embed.h5ad"
    if embed_path.exists():
        RUN = candidate
        break
if RUN is None:
    raise FileNotFoundError(f"No embed found for any of {CANDIDATES}")

PSEUDO_DIR = S.SCRATCH / "pseudobulk"
SCORE_CUT  = 0.10   # OOD traversal score threshold (same as 02/04)
MAX_GENES  = 100    # cap per direction before passing to ORA
CACHE_DIR  = S.SCRATCH / "annotation"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"run: {RUN}")
print(f"embed: {embed_path}")


## §2 — Data

In [ ]:
embed = ad.read_h5ad(embed_path, backed="r")
gene_names = np.asarray(embed.uns["gene_names"]).astype(str)

print(f"{embed.n_obs:,} cells × {embed.n_vars} factors")
print(f"{len(gene_names):,} genes in the model universe")


In [ ]:
# Traversal scores: genes × directions, one column per live direction.
# OOD_combined_positive / _negative store the per-gene decoder response
# when moving each factor in the positive / negative direction from the
# prior mean. Directions flagged vanished are excluded. Ordering and
# naming mirror 02_explore §3 and 04_progression §7 exactly.
effect = np.concatenate([embed.varm["OOD_combined_positive"],
                         embed.varm["OOD_combined_negative"]])
gene_info = (
    pd.concat([embed.var.assign(direction="+"), embed.var.assign(direction="-")])
    .assign(title=lambda df: df["title"] + df["direction"])
    .reset_index(drop=True)
)
gone = np.where(
    gene_info["direction"] == "+",
    gene_info["vanished_positive_direction"],
    gene_info["vanished_negative_direction"],
)
live = gene_info[~gone].sort_values(["order", "direction"])["title"]

traversal = pd.DataFrame(
    effect, columns=pd.Index(gene_names), index=gene_info["title"]
).loc[live].T

print(f"{traversal.shape[0]} genes × {traversal.shape[1]} live directions")


In [ ]:
# Supertype pseudobulk: donor × region × supertype means per direction.
within_act = pd.read_parquet(PSEUDO_DIR / "within_activity.parquet")
SUPERTYPES = within_act.index.get_level_values("Supertype").unique().sort_values().tolist()
DIRECTIONS = traversal.columns.tolist()

print(f"{within_act.shape[0]:,} units · {len(SUPERTYPES)} supertypes · {len(DIRECTIONS)} directions")


## §3 — Supertype specificity

Mean activation per direction grouped by supertype across all donor × region units.
A direction with a high mean for one supertype and near-zero for the rest is
supertype-specific; one with similar means across supertypes is pan-SST.


In [ ]:
# Mean activation per (supertype, direction). ReLU-split values are already
# non-negative (pseudobulk of abs(relu(x))); taking the group mean gives a
# supertype × direction profile.
profile = (
    within_act[DIRECTIONS]
    .groupby(level="Supertype")
    .mean()
)
print(f"Supertype mean-activation matrix: {profile.shape}")

# Z-score down each direction column so that the heatmap shows *relative*
# enrichment rather than raw magnitude (which is dominated by overall factor
# activity level).
profile_z = (profile - profile.mean()) / profile.std(ddof=0).replace(0, np.nan)


In [ ]:
# Heatmap: supertypes (rows) × directions (columns), z-scored.
# Cluster both axes to group co-active supertypes and co-expressed directions.
g = sns.clustermap(
    profile_z,
    figsize=(max(14, len(DIRECTIONS) * 0.14), max(6, len(SUPERTYPES) * 0.45)),
    cmap="RdBu_r", center=0, vmin=-2.5, vmax=2.5,
    dendrogram_ratio=(0.08, 0.10),
    cbar_pos=(1.01, 0.3, 0.02, 0.4),
    linewidths=0,
    xticklabels=True, yticklabels=True,
)
g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(), fontsize=5, rotation=90
)
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(), fontsize=8, rotation=0
)
g.figure.suptitle("Supertype × direction activity (z-scored)", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Primary supertype: for each direction, the supertype with the highest mean
# activation; and the specificity score — the ratio of the top supertype's
# mean to the sum across all supertypes (0 = pan-SST, 1 = exclusively one
# supertype). Also report the top-3 supertypes.
def _top_supertypes(series, n=3):
    return ", ".join(series.nlargest(n).index.tolist())

supertype_annotation = pd.DataFrame({
    "primary_supertype": profile.idxmax(),
    "primary_mean":      profile.max().round(3),
    "specificity":       (profile.max() / profile.sum().replace(0, np.nan)).round(3),
    "top3_supertypes":   profile.apply(_top_supertypes),
}).sort_values("specificity", ascending=False)

print("Most supertype-specific directions (top 20):")
supertype_annotation.head(20)


In [ ]:
# How many directions does each supertype "own" (highest mean activation)?
owned = supertype_annotation["primary_supertype"].value_counts()
print("Directions owned by each supertype (primary_supertype):")
print(owned.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
owned.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Directions (primary supertype)")
ax.set_title("Which SST supertype has the highest mean activation per direction?")
plt.tight_layout()
plt.show()


## §4 — Regional patterns

Two views of regional structure per direction:

1. **Activity**: mean pseudobulk activation per brain region — where is each direction expressed?
2. **CPS slope by region**: the hierarchical model's per-region slope posterior means
   (`mu_beta + u_region`) from the ADVI fit — where does the direction *change* with pathology?

Directional `sigma_beta_mean` (between-region SD of slopes) from §6 of
`archive/06_hierarchical_bayes.ipynb` provides a single heterogeneity summary per direction.


In [ ]:
# Load donor × region pseudobulk (not the within-supertype one).
activity = pd.read_parquet(PSEUDO_DIR / "activity.parquet")

# Region mean activity: average across donors within each region.
# The pseudobulk index is (Donor ID, Brain Region); groupby collapses donors.
region_act = activity.groupby(level="Brain Region").mean()
REGIONS = region_act.index.tolist()
print(f"Region mean-activation matrix: {region_act.shape}")

region_act_z = (region_act - region_act.mean()) / region_act.std(ddof=0).replace(0, np.nan)


In [ ]:
# Heatmap: regions (rows) × directions (columns), z-scored.
g = sns.clustermap(
    region_act_z,
    figsize=(max(14, len(DIRECTIONS) * 0.14), 5),
    cmap="RdBu_r", center=0, vmin=-2.5, vmax=2.5,
    dendrogram_ratio=(0.12, 0.08),
    cbar_pos=(1.01, 0.3, 0.02, 0.4),
    linewidths=0,
    xticklabels=True, yticklabels=True,
)
g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(), fontsize=5, rotation=90
)
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(), fontsize=9, rotation=0
)
g.figure.suptitle("Brain region × direction activity (z-scored)", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Regional specificity: same formula as supertype specificity.
# A high score means the direction is concentrated in one region.
def _top_regions(series, n=3):
    return ", ".join(series.nlargest(n).index.tolist())

region_annotation = pd.DataFrame({
    "primary_region":   region_act.idxmax(),
    "primary_mean":     region_act.max().round(3),
    "region_specificity": (region_act.max() / region_act.sum().replace(0, np.nan)).round(3),
    "top3_regions":     region_act.apply(_top_regions),
}).sort_values("region_specificity", ascending=False)

print("Most region-specific directions (top 20):")
region_annotation.head(20)


In [ ]:
# Directions owned per region (primary region = highest mean activation).
owned_region = region_annotation["primary_region"].value_counts()
print("Directions owned by each brain region:")
print(owned_region.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
owned_region.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Directions (primary region)")
ax.set_title("Which brain region has the highest mean activation per direction?")
plt.tight_layout()
plt.show()


In [ ]:
# Per-region CPS slopes from the ADVI fit (06_hierarchical_bayes §4).
# Columns: beta_<Region>_mean = mu_beta + u_region for each region.
vi = pd.read_parquet(PSEUDO_DIR / "bayes_vi_CPS_Local.parquet")
beta_cols = [c for c in vi.columns if c.startswith("beta_") and c.endswith("_mean")]
region_slopes = vi[beta_cols].copy()
region_slopes.columns = [c.replace("beta_", "").replace("_mean", "") for c in beta_cols]
# rows = directions, columns = regions

print(f"Per-region slope matrix: {region_slopes.shape}")


In [ ]:
# Heatmap: directions (rows) × regions (columns), per-region CPS slope.
# Positive = activity increases with CPS in that region; negative = decreases.
# Sort rows by pan-region slope |mu_beta_mean| so the strongest effects are visible.
row_order = vi["mu_beta_mean"].abs().sort_values(ascending=False).index
region_slopes_ord = region_slopes.reindex(row_order)

fig, ax = plt.subplots(figsize=(6, max(10, len(DIRECTIONS) * 0.14)))
vabs = region_slopes.abs().quantile(0.95).max()
sns.heatmap(
    region_slopes_ord,
    cmap="RdBu_r", center=0, vmin=-vabs, vmax=vabs,
    linewidths=0.3, linecolor="0.9",
    cbar_kws={"label": "Per-region CPS slope (μ_β + u_r)", "shrink": 0.4},
    ax=ax,
    xticklabels=True, yticklabels=True,
)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=5)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=9, rotation=45, ha="right")
ax.set_title("Per-region CPS slope (ADVI posterior mean)\nDirections sorted by |pan slope|", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# sigma_beta_mean: between-region SD of the CPS slope (from 06 §6).
# Small sigma_beta → pan-regional effect; large → heterogeneous across regions.
vi = pd.read_parquet(PSEUDO_DIR / "bayes_vi_CPS_Local.parquet")
beta_cols = [c for c in vi.columns if c.startswith("beta_") and c.endswith("_mean")]
region_slopes = vi[beta_cols].copy()
region_slopes.columns = [c.replace("beta_", "").replace("_mean", "") for c in beta_cols]

sigma_beta = vi["sigma_beta_mean"].sort_values(ascending=False)
print("Top 15 directions by between-region slope heterogeneity (sigma_beta):")
print(sigma_beta.head(15).round(3).to_string())


## §5 — Gene programs

Top genes per direction by OOD traversal score and their gene-set enrichment.


In [ ]:
# Top genes table: rank × direction (first 20 directions shown).
top_genes = pd.DataFrame([
    {"direction": d, "rank": r, "gene": g, "score": s}
    for d in DIRECTIONS
    for r, (g, s) in enumerate(traversal[d].nlargest(10).items(), start=1)
])
top_genes_wide = top_genes.pivot(index="rank", columns="direction", values="gene")
print("Top 10 genes per direction (first 20 directions shown):")
top_genes_wide.iloc[:, :20]


## §6 — Gene-set enrichment (ORA)

Over-representation analysis for all live directions against four collections.
Gene lists are capped at `MAX_GENES` and require `SCORE_CUT = 0.10`. Results
are cached to parquet so re-runs are instant.


In [ ]:
COLLECTIONS = {
    "Hallmark": "MSigDB_Hallmark_2020",
    "GO BP":    "GO_Biological_Process_2023",
    "GO MF":    "GO_Molecular_Function_2023",
    "GO CC":    "GO_Cellular_Component_2023",
}
GENESETS = S.SCRATCH / "genesets"

absent = [n for n in COLLECTIONS.values() if not (GENESETS / f"{n}.gmt").exists()]
if absent:
    fetch = "\n".join(
        f"  curl -sfL 'https://maayanlab.cloud/Enrichr/geneSetLibrary"
        f"?mode=text&libraryName={n}' -o {GENESETS / f'{n}.gmt'}" for n in absent
    )
    raise FileNotFoundError(f"missing {len(absent)} GMT file(s):\n{fetch}")


In [ ]:
ENRICH_CACHE = CACHE_DIR / "ora_all_directions.parquet"

if ENRICH_CACHE.exists():
    enrich_df = pd.read_parquet(ENRICH_CACHE)
    print(f"loaded {len(enrich_df):,} rows from cache")
else:
    background = gene_names.tolist()

    # Pre-parse each GMT once; pass as dict to gseapy to avoid re-reading per direction.
    parsed_sets = {}
    for label, name in COLLECTIONS.items():
        sets = {}
        for line in (GENESETS / f"{name}.gmt").read_text().splitlines():
            fields = line.split("\t")
            sets[fields[0]] = [g for g in fields[2:] if g]
        parsed_sets[label] = sets
        print(f"  {label}: {len(sets)} gene sets loaded")

    gene_lists = {}
    for direction in DIRECTIONS:
        genes = (
            traversal[direction]
            .pipe(lambda s: s[s >= SCORE_CUT])
            .nlargest(MAX_GENES)
            .index.astype(str)
            .tolist()
        )
        if len(genes) >= 5:
            gene_lists[direction] = genes
        else:
            logger.info("%s: %d genes at score >= %.2f, skipped", direction, len(genes), SCORE_CUT)

    print(f"\n{len(gene_lists)}/{len(DIRECTIONS)} directions with >= 5 genes above score cut")

    frames = []
    for label, sets in parsed_sets.items():
        for direction, genes in gene_lists.items():
            res = gp.enrich(
                gene_list=genes,
                gene_sets=sets,
                background=background,
                outdir=None,
                verbose=False,
            ).results
            frames.append(res.assign(collection=label, direction=direction))
        print(f"  {label}: done ({len(gene_lists)} directions)")

    enrich_df = pd.concat(frames, ignore_index=True)
    enrich_df.to_parquet(ENRICH_CACHE)
    print(f"\nsaved {len(enrich_df):,} rows to {ENRICH_CACHE}")

significant = enrich_df[enrich_df["Adjusted P-value"] < 0.05].copy()
print(f"\nSignificant direction-term pairs at adj p < 0.05: {len(significant):,}")
print(significant.groupby("collection")["direction"].nunique().rename("directions with ≥1 hit").to_string())


In [ ]:
# Coverage summary: for each collection, what fraction of a direction's gene
# list does the collection's universe cover? Low coverage = collection is the
# wrong tool for this tissue, not a biological finding.
universe_by_collection = {
    label: set().union(*[set(g) for g in sets.values()])
    for label, sets in parsed_sets.items()
} if "parsed_sets" in dir() else None  # only when we ran the enrichment above

summary_rows = []
for label in COLLECTIONS:
    sub = enrich_df[enrich_df["collection"] == label]
    n_sets = sub["Term"].nunique()
    n_hits = significant[significant["collection"] == label]["direction"].nunique()
    summary_rows.append({"collection": label, "gene sets": n_sets,
                         "directions with ≥1 hit": n_hits})
summary = pd.DataFrame(summary_rows).set_index("collection")
summary


In [ ]:
# Bubble plot: top 3 terms per direction × collection (same style as 04).
PER_DIRECTION = 3

panels = {}
for label in COLLECTIONS:
    sub = significant[significant["collection"] == label]
    if sub.empty:
        continue
    top_terms = (
        sub.sort_values("Adjusted P-value")
        .groupby("direction", observed=True)
        .head(PER_DIRECTION)
    )
    terms = top_terms["Term"].drop_duplicates().tolist()
    panels[label] = (sub[sub["Term"].isin(terms)], terms)

if not panels:
    print("nothing significant in any collection")
else:
    # Order directions by their primary supertype then by direction label.
    order = (
        supertype_annotation.reindex(DIRECTIONS)
        .sort_values(["primary_supertype", "specificity"], ascending=[True, False])
        .index.tolist()
    )
    order = [d for d in order if d in significant["direction"].values]

    rows = [len(terms) for _, terms in panels.values()]
    fig, axes = plt.subplots(
        len(panels), 1, squeeze=False, layout="constrained",
        figsize=(max(10, len(order) * 0.25 + 6), 0.32 * sum(rows) + 1.4 * len(panels)),
        gridspec_kw={"height_ratios": rows},
    )
    limits = -np.log10(significant["Adjusted P-value"])
    for ax, (label, (sub, terms)) in zip(axes[:, 0], panels.items()):
        x_map = {d: i for i, d in enumerate(order)}
        present = sub[sub["direction"].isin(x_map)]
        x = [x_map[d] for d in present["direction"]]
        y = [terms.index(t) for t in present["Term"]]
        sc = ax.scatter(
            x, y,
            s=present["Overlap"].str.split("/").str[0].astype(int) * 10,
            c=-np.log10(present["Adjusted P-value"]),
            cmap="rocket_r", vmin=limits.min(), vmax=limits.max(),
            edgecolor="0.3", linewidth=0.4,
        )
        ax.set_xticks(range(len(order)))
        if ax is axes[-1, 0]:
            ax.set_xticklabels(order, rotation=90, fontsize=6)
        else:
            ax.set_xticklabels([])
        ax.set_yticks(range(len(terms)))
        ax.set_yticklabels(terms, fontsize=7)
        ax.set_xlim(-0.8, len(order) - 0.2)
        ax.set_title(label, fontsize=9, loc="left")
        fig.colorbar(sc, ax=ax, label="-log10(adj p)")
    fig.suptitle("Gene-set enrichment across all live directions", y=1.01)
    plt.show()


## §7 — Summary annotation table

One row per direction: primary supertype, top genes, top enrichment term per
collection. Join with CPS slope from §4 of `04_progression.ipynb` if available.


In [ ]:
def _best_term(direction, collection):
    sub = significant[(significant["direction"] == direction) &
                      (significant["collection"] == collection)]
    if sub.empty:
        return ""
    return sub.sort_values("Adjusted P-value").iloc[0]["Term"]


annotation = pd.DataFrame(
    index=pd.Index(DIRECTIONS, name="direction"),
    data={
        "primary_supertype": supertype_annotation.reindex(DIRECTIONS)["primary_supertype"],
        "specificity":       supertype_annotation.reindex(DIRECTIONS)["specificity"],
        "top3_supertypes":   supertype_annotation.reindex(DIRECTIONS)["top3_supertypes"],
        "n_genes_above_cut": [
            int((traversal[d] >= SCORE_CUT).sum()) for d in DIRECTIONS
        ],
        "top_genes": [
            ", ".join(traversal[d].nlargest(8).index.tolist()) for d in DIRECTIONS
        ],
        "top_GO_BP": [_best_term(d, "GO BP")  for d in DIRECTIONS],
        "top_GO_MF": [_best_term(d, "GO MF")  for d in DIRECTIONS],
        "top_GO_CC": [_best_term(d, "GO CC")  for d in DIRECTIONS],
        "top_Hallmark": [_best_term(d, "Hallmark") for d in DIRECTIONS],
    },
)

# Attach composite CPS slope: prefer Bayesian VI result (mu_beta_mean from 06),
# fall back to mixed-model coefficient (coef from 04 if cached separately).
vi_cache = PSEUDO_DIR / "bayes_vi_CPS_Local.parquet"
mm_cache = PSEUDO_DIR / "trend_CPS_Local.parquet"
if vi_cache.exists():
    vi = pd.read_parquet(vi_cache)
    annotation["CPS_slope_bayes"] = vi["mu_beta_mean"].reindex(annotation.index)
    annotation["CPS_p_positive"]  = vi["p_positive"].reindex(annotation.index)
    print("Bayesian VI slopes attached from 06 cache")
elif mm_cache.exists():
    slopes = pd.read_parquet(mm_cache)
    annotation["CPS_slope"] = slopes["coef"].reindex(annotation.index)
    print("Mixed-model slopes attached from 04 cache")
else:
    print("no slope cache found -- run 04_progression.ipynb or archive/06_hierarchical_bayes.ipynb first")


# Attach regional annotation columns (computed in §4).
annotation["primary_region"]     = region_annotation.reindex(annotation.index)["primary_region"]
annotation["region_specificity"] = region_annotation.reindex(annotation.index)["region_specificity"]
annotation["top3_regions"]       = region_annotation.reindex(annotation.index)["top3_regions"]
annotation["sigma_beta_CPS"]     = vi["sigma_beta_mean"].reindex(annotation.index)
annotation["mu_beta_CPS"]        = vi["mu_beta_mean"].reindex(annotation.index)

annotation_cache = CACHE_DIR / "direction_annotation.parquet"
annotation.to_parquet(annotation_cache)
print(f"written to {annotation_cache}")
print(f"\n{len(annotation)} directions annotated")
annotation.head(20)


In [ ]:
# Save a TSV for easy sharing.
annotation.to_csv(CACHE_DIR / "direction_annotation.tsv", sep="\t")
print("TSV written to", CACHE_DIR / "direction_annotation.tsv")


## §8 — Summary

- **Supertype specificity**: `specificity` = max supertype mean / sum across supertypes.
  High values (→ 1) indicate a direction active almost exclusively in one supertype;
  low values indicate pan-SST programs.
- **Gene programs**: OOD traversal scores ≥ 0.10 are the biologically informative genes
  (same threshold as the DRVI tutorial and `04_progression.ipynb`).
- **Enrichment**: ORA against four collections using the model's own gene universe as
  background. Hallmark coverage is typically low (built from bulk cancer/immune data);
  GO BP is the primary source of terms. A direction with no enrichment at adj p < 0.05
  is not necessarily uninterpretable — it may encode a cell-type identity or rare
  program without a matching GO term.
- Results are cached to `/scratch/sst-drvi/annotation/` and can be read by downstream
  notebooks without re-running this one.


## §9 — Compositional context: scCODA overlap

Which annotated directions correspond to supertypes that are compositionally
depleted with AD progression?

**Logic**: scCODA (CPS_Local, global fit) identifies SST supertypes whose share
of the neuronal composition changes credibly with pathology. A direction whose
primary supertype is credibly depleted tells a *cell-loss* story — the program
disappears because the cells carrying it are lost. A direction primary to a
depleted supertype that nonetheless *increases* with CPS suggests a
cell-autonomous compensatory response in the surviving cells.

The credibility threshold is `S.SCCODA_INCLUSION_THRESHOLD` (median posterior
inclusion probability across reference-cell-type choices ≥ 0.83).


In [ ]:
# scCODA global fit: median effect and median inclusion probability across the
# 173 reference-cell-type choices, restricted to SST supertypes and the
# CPS_Local covariate. The sweep is read in chunks to avoid materialising the
# full 180 MB neuronal results file.
SWEEP_COLUMNS = ["Region", "Covariate", "Cell Type", "Final Parameter",
                 "Inclusion probability", "Reference Cell Type"]

chunks = []
for path in sorted(S.CPS_LOCAL_DIR.glob(S.SCCODA_RESULTS_GLOB)):
    for chunk in pd.read_csv(path, usecols=SWEEP_COLUMNS, chunksize=500_000):
        chunk = chunk[(chunk["Covariate"] == "CPS_Local")
                      & chunk["Cell Type"].str.startswith("Sst_")]
        if len(chunk):
            chunks.append(chunk)

swept = pd.concat(chunks, ignore_index=True)
swept = swept[swept["Cell Type"] != swept["Reference Cell Type"]]

global_fit = (
    swept[swept["Region"] == "Global"]
    .groupby("Cell Type", observed=True)
    .agg(sccoda_effect   =("Final Parameter",    "median"),
         sccoda_inclusion=("Inclusion probability","median"))
    .astype(float)
)
global_fit["credible"] = global_fit["sccoda_inclusion"] >= S.SCCODA_INCLUSION_THRESHOLD
credible_sts = set(global_fit[global_fit["credible"]].index)

print(f"Credible SST supertypes ({len(credible_sts)}):")
print(global_fit[global_fit["credible"]]
      .sort_values("sccoda_effect")
      .round(3).to_string())


In [ ]:
# Join: attach scCODA credibility to each direction via its primary supertype.
link = annotation.assign(
    sccoda_effect   = lambda d: d["primary_supertype"].map(global_fit["sccoda_effect"]).astype(float),
    sccoda_inclusion= lambda d: d["primary_supertype"].map(global_fit["sccoda_inclusion"]).astype(float),
    st_credible     = lambda d: d["primary_supertype"].isin(credible_sts),
    abs_cps         = lambda d: d["CPS_slope_bayes"].abs(),
)

cps     = link["CPS_slope_bayes"].astype(float)
p_pos   = link["CPS_p_positive"].astype(float)
strong  = (p_pos < 0.05) | (p_pos > 0.95)   # Bayesian P away from 0.5

co_lost   = link["st_credible"] & (cps < 0) & (p_pos < 0.05)
counter   = link["st_credible"] & (cps > 0) & (p_pos > 0.95)
credible_strong = link["st_credible"] & strong

print(f"Directions with a credible-scCODA primary supertype: {link['st_credible'].sum()}")
print(f"  Strong CPS association (P<0.05 or >0.95): {credible_strong.sum()}")
print(f"    Co-lost  (direction ↓ as supertype depleted): {co_lost.sum()}")
print(f"    Counter  (direction ↑ as supertype depleted): {counter.sum()}")
print(f"  Uncertain CPS direction: {(link['st_credible'] & ~strong).sum()}")
print()
print(f"Strong CPS directions with no credible-scCODA primary supertype: "
      f"{(strong & ~link['st_credible']).sum()}  (candidate cell-intrinsic)")


In [ ]:
# Scatter: CPS slope (x) vs scCODA effect (y) for all directions with a
# primary supertype in the scCODA table. Only supertypes that appear in
# global_fit are shown; directions whose primary supertype was not modelled
# are dropped (they have NaN sccoda_effect). Marker size encodes supertype
# specificity; colour encodes the credibility of the scCODA call.
have_sccoda = link["sccoda_effect"].notna()
sub = link[have_sccoda].copy()
sub_cps  = sub["CPS_slope_bayes"].astype(float)
sub_pp   = sub["CPS_p_positive"].astype(float)
sub_efx  = sub["sccoda_effect"].astype(float)
sub_incl = sub["sccoda_inclusion"].astype(float)

color = np.where(sub_incl >= S.SCCODA_INCLUSION_THRESHOLD, "#c0392b", "#7f8c8d")
size  = sub["specificity"].astype(float) * 120 + 10

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(sub_cps, sub_efx, c=color, s=size,
                alpha=0.75, edgecolors="0.3", linewidths=0.4)

# Label the co-lost and counter quadrant points with strong CPS
strong_mask = (sub_pp < 0.05) | (sub_pp > 0.95)
for direction, row in sub[strong_mask].iterrows():
    ax.annotate(direction, (row["CPS_slope_bayes"], row["sccoda_effect"]),
                fontsize=6.5, ha="center", va="bottom",
                xytext=(0, 3), textcoords="offset points")

ax.axhline(0, color="0.4", lw=0.8, ls="--")
ax.axvline(0, color="0.4", lw=0.8, ls="--")
ax.set_xlabel("Bayesian CPS slope (μ_β, composite)", fontsize=10)
ax.set_ylabel("scCODA global effect (median over references)", fontsize=10)
ax.set_title("Direction CPS slope vs supertype compositional change\n"
             "red = credible scCODA effect  ·  size = supertype specificity", fontsize=9)

# Legend patches
from matplotlib.patches import Patch
legend_els = [Patch(fc="#c0392b", ec="0.3", label=f"Credible (incl ≥ {S.SCCODA_INCLUSION_THRESHOLD})"),
              Patch(fc="#7f8c8d", ec="0.3", label="Not credible")]
ax.legend(handles=legend_els, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Co-lost directions: the program shrinks alongside the supertype.
cols = ["primary_supertype", "specificity", "CPS_slope_bayes", "CPS_p_positive",
        "sccoda_effect", "sccoda_inclusion", "top_GO_BP"]
co_lost_tbl = (link[co_lost][cols]
               .sort_values("CPS_slope_bayes")
               .round(3))
print(f"Co-lost directions ({len(co_lost_tbl)}):")
co_lost_tbl


In [ ]:
# Counter-directional: the program increases as the supertype shrinks.
counter_tbl = (link[counter][cols]
               .sort_values("CPS_slope_bayes", ascending=False)
               .round(3))
print(f"Counter-directional directions ({len(counter_tbl)}):")
counter_tbl


In [ ]:
# Write the extended annotation table (adds sccoda columns).
link_cache = CACHE_DIR / "direction_annotation_sccoda.parquet"
link.to_parquet(link_cache)
link.to_csv(str(link_cache).replace(".parquet", ".tsv"), sep="\t")
print(f"Extended annotation written to {link_cache}")

# Quick summary sentence.
n_strong_noncell_loss = (strong & ~link["st_credible"]).sum()
n_co_lost = co_lost.sum()
n_counter = counter.sum()
print(f"\n{n_co_lost} directions co-lost with a depleted supertype (cell-loss candidates)")
print(f"{n_counter} directions increase as their primary supertype is depleted (cell-intrinsic candidates)")
print(f"{n_strong_noncell_loss} strong-CPS directions have no credible-scCODA primary supertype (cell-intrinsic candidates)")


In [ ]:
from session_info2 import session_info

session_info(dependencies=True)
